In [2]:
import os
import sys
import numpy as np
import torch
import pandas as pd

# 让notebook能import MacroHFT
ROOT = '/home/ustc1958/lxy/129/MacroHFT'
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from model.net import subagent

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('device =', DEVICE)

# ==========================================================
# 配置：test 高层 epoch_1 + test 低层 six subagents
# ==========================================================
HIGH_TEST_E1 = '/home/ustc1958/lxy/129/MacroHFT/result/high_level/ETHUSDT/131test/seed_12345/epoch_1/trained_model.pkl'
LOW_TEST_BASE = '/home/ustc1958/lxy/129/MacroHFT/result/low_level/ETHUSDT/131testbest_model'

names = ['slope_1','slope_2','slope_3','vol_1','vol_2','vol_3']

# ==========================================================
# 1) 读取特征列表
# ==========================================================
tech_indicator_list = np.load('/home/ustc1958/lxy/129/MacroHFT/data/feature_list/single_features.npy', allow_pickle=True).tolist()
tech_indicator_list_trend = np.load('/home/ustc1958/lxy/129/MacroHFT/data/feature_list/trend_features.npy', allow_pickle=True).tolist()
N_STATE_1 = len(tech_indicator_list)
N_STATE_2 = len(tech_indicator_list_trend)
print('N_STATE_1 =', N_STATE_1, 'N_STATE_2 =', N_STATE_2)

# ==========================================================
# 2) 抽样 state：优先使用“低层验证集 df_*.feather”，以贴近你说的 val 表现
#    - 分别从 slope/vol 的 label_1/2/3 各抽取一小批
#    - 若 val df 不存在/路径不一致，可把 FALLBACK 改回 whole/train.feather
# ==========================================================
VAL_DIR = '/home/ustc1958/lxy/129/MacroHFT/data/ETHUSDT/val'
VAL_FEATHER_DIR = VAL_DIR  # 这里约定 val 目录下包含 df_*.feather

B_PER_GROUP = 96   # 每个 (clf,label,df) 抽样量（总量会自动裁剪到可用行数）
MAX_DF_PER_GROUP = 1  # 每组取多少个 df_*.feather（1 足够做诊断）

FALLBACK_WHOLE = '/home/ustc1958/lxy/129/MacroHFT/data/ETHUSDT/whole/train.feather'

# 低层 val 标签文件（用于拿 df id 列表）
SLOPE_VAL_LABELS = os.path.join(VAL_DIR, 'slope_labels.pkl')
VOL_VAL_LABELS = os.path.join(VAL_DIR, 'vol_labels.pkl')

@torch.no_grad()
def _sample_from_df_feather(df_path, n):
    _df = pd.read_feather(df_path)
    n = min(n, len(_df))
    ridx = np.random.choice(len(_df), size=n, replace=False)
    s1 = torch.tensor(_df.loc[ridx, tech_indicator_list].to_numpy(np.float32), device=DEVICE)
    s2 = torch.tensor(_df.loc[ridx, tech_indicator_list_trend].to_numpy(np.float32), device=DEVICE)
    return s1, s2

# 构造 val 采样
state_list, trend_list, tag_list = [], [], []

try:
    import pickle
    with open(SLOPE_VAL_LABELS, 'rb') as f:
        slope_idx = pickle.load(f)
    with open(VOL_VAL_LABELS, 'rb') as f:
        vol_idx = pickle.load(f)

    # label 为 1/2/3
    for clf, idx_map in [('slope', slope_idx), ('vol', vol_idx)]:
        for label in [1, 2, 3]:
            df_list = idx_map[label]
            # 只取前 MAX_DF_PER_GROUP 个 df
            for dfid in df_list[:MAX_DF_PER_GROUP]:
                df_path = os.path.join(VAL_FEATHER_DIR, f'df_{dfid}.feather')
                s1, s2 = _sample_from_df_feather(df_path, B_PER_GROUP)
                state_list.append(s1)
                trend_list.append(s2)
                tag_list.append(f'{clf}_label{label}_df{dfid}')

    state = torch.cat(state_list, dim=0)
    state_trend = torch.cat(trend_list, dim=0)
    B = state.shape[0]
    previous_action = torch.zeros((B,), dtype=torch.long, device=DEVICE)

    print('[val-sample] groups=', len(tag_list), 'total B=', B)
    print('tags:', tag_list)

except Exception as e:
    print('[val-sample] failed -> fallback to whole/train.feather, err:', repr(e))
    df = pd.read_feather(FALLBACK_WHOLE)
    B = 512
    ridx = np.random.choice(len(df), size=B, replace=False)
    state = torch.tensor(df.loc[ridx, tech_indicator_list].to_numpy(np.float32), device=DEVICE)
    state_trend = torch.tensor(df.loc[ridx, tech_indicator_list_trend].to_numpy(np.float32), device=DEVICE)
    previous_action = torch.zeros((B,), dtype=torch.long, device=DEVICE)

print('state', tuple(state.shape), 'state_trend', tuple(state_trend.shape), 'previous_action', tuple(previous_action.shape))

# ==========================================================
# 3) 加载 test 低层六个 subagent，并统计单体 Q 分布
# ==========================================================
N_ACTION = 2
LOW_HIDDEN = 64

def build_subagents():
    out = {
        'slope_1': subagent(N_STATE_1, N_STATE_2, N_ACTION, LOW_HIDDEN).to(DEVICE),
        'slope_2': subagent(N_STATE_1, N_STATE_2, N_ACTION, LOW_HIDDEN).to(DEVICE),
        'slope_3': subagent(N_STATE_1, N_STATE_2, N_ACTION, LOW_HIDDEN).to(DEVICE),
        'vol_1': subagent(N_STATE_1, N_STATE_2, N_ACTION, LOW_HIDDEN).to(DEVICE),
        'vol_2': subagent(N_STATE_1, N_STATE_2, N_ACTION, LOW_HIDDEN).to(DEVICE),
        'vol_3': subagent(N_STATE_1, N_STATE_2, N_ACTION, LOW_HIDDEN).to(DEVICE),
    }
    for m in out.values():
        m.eval()
    return out


def load_low_weights(agents, base_dir: str):
    mapping = {
        'slope_1': os.path.join(base_dir, 'slope', '1', 'best_model.pkl'),
        'slope_2': os.path.join(base_dir, 'slope', '2', 'best_model.pkl'),
        'slope_3': os.path.join(base_dir, 'slope', '3', 'best_model.pkl'),
        'vol_1': os.path.join(base_dir, 'vol', '1', 'best_model.pkl'),
        'vol_2': os.path.join(base_dir, 'vol', '2', 'best_model.pkl'),
        'vol_3': os.path.join(base_dir, 'vol', '3', 'best_model.pkl'),
    }
    for n, net in agents.items():
        net.load_state_dict(torch.load(mapping[n], map_location=DEVICE))

agents_low = build_subagents()
load_low_weights(agents_low, LOW_TEST_BASE)
print('loaded test low-level agents')

@torch.no_grad()
def stack_q(agents):
    qs = []
    for n in names:
        q = agents[n](state, state_trend, previous_action)  # [B,2]
        qs.append(q.unsqueeze(1))
    return torch.cat(qs, dim=1)  # [B,6,2]

@torch.no_grad()
def q_stats_per_agent(qs_b62):
    rows = []
    for i, n in enumerate(names):
        q = qs_b62[:, i, :]
        gap = (q[:, 0] - q[:, 1]).abs()
        rows.append({
            'agent': n,
            'q_mean': float(q.mean().item()),
            'q_std': float(q.std().item()),
            'q_min': float(q.min().item()),
            'q_max': float(q.max().item()),
            'gap_mean': float(gap.mean().item()),
            'gap_p95': float(torch.quantile(gap, 0.95).item()),
            'gap_max': float(gap.max().item()),
        })
    return pd.DataFrame(rows).sort_values('gap_max', ascending=False)

qs = stack_q(agents_low)
print('\n' + '='*100)
print('Low-level Q stats on (val-like) states')
print('='*100)
display(q_stats_per_agent(qs))

# ==========================================================
# 4) 加载 test 高层 epoch_1 gating，计算每个样本的 6 维权重 w
# ==========================================================
ckpt = torch.load(HIGH_TEST_E1, map_location=DEVICE)

state_dict = None
if isinstance(ckpt, dict) and 'embedding.weight' in ckpt and 'net.0.weight' in ckpt:
    state_dict = ckpt
elif isinstance(ckpt, dict):
    for key in ['state_dict', 'model', 'hyperagent', 'policy', 'net', 'actor']:
        if key in ckpt and isinstance(ckpt[key], dict) and 'embedding.weight' in ckpt[key]:
            state_dict = ckpt[key]
            break

if state_dict is None:
    raise RuntimeError('无法从 HIGH_TEST_E1 解析 state_dict，请检查 ckpt keys')

N_ACTION_CKPT = int(state_dict['embedding.weight'].shape[0])
EMB_DIM = int(state_dict['embedding.weight'].shape[1])
HIDDEN = int(state_dict['net.0.weight'].shape[0])
IN_DIM = int(state_dict['net.0.weight'].shape[1])
N_AGENTS = int(state_dict['net.2.weight'].shape[0])

print('test high gating inferred: N_ACTION=', N_ACTION_CKPT, 'EMB_DIM=', EMB_DIM, 'HIDDEN=', HIDDEN, 'IN_DIM=', IN_DIM, 'N_AGENTS=', N_AGENTS)

class GatingNet(torch.nn.Module):
    def __init__(self, n_action, emb_dim, hidden, in_dim, n_agents):
        super().__init__()
        self.embedding = torch.nn.Embedding(n_action, emb_dim)
        self.net = torch.nn.Sequential(
            torch.nn.Linear(in_dim, hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden, n_agents),
        )

    def forward(self, x_in):
        logits = self.net(x_in)
        w = torch.softmax(logits, dim=1)
        return w, logits

hyper = GatingNet(N_ACTION_CKPT, EMB_DIM, HIDDEN, IN_DIM, N_AGENTS).to(DEVICE)

cur = hyper.state_dict()
filtered = {k: v for k, v in state_dict.items() if k in cur and v.shape == cur[k].shape}
hyper.load_state_dict({**cur, **filtered}, strict=True)
hyper.eval()

@torch.no_grad()
def build_x_in(state, state_trend, previous_action):
    emb = hyper.embedding(previous_action)
    feats = torch.cat([state, state_trend, emb], dim=1)
    # 注意：这里仍是“截断/补零”到 IN_DIM，只用于诊断 gating 是否塌缩
    if feats.shape[1] >= IN_DIM:
        return feats[:, :IN_DIM]
    pad = torch.zeros((feats.shape[0], IN_DIM - feats.shape[1]), device=feats.device, dtype=feats.dtype)
    return torch.cat([feats, pad], dim=1)

x_in = build_x_in(state, state_trend, previous_action)
w, logits = hyper(x_in)
print('w shape:', tuple(w.shape), 'logits shape:', tuple(logits.shape))

@torch.no_grad()
def weight_report(w_b6):
    eps = 1e-12
    ent = -(w_b6.clamp_min(eps) * w_b6.clamp_min(eps).log()).sum(dim=1)
    wmax = w_b6.max(dim=1).values
    top1 = w_b6.argmax(dim=1)
    counts = torch.bincount(top1, minlength=6).float()
    frac = counts / counts.sum()

    per = []
    for i, n in enumerate(names):
        wi = w_b6[:, i]
        per.append({
            'agent': n,
            'w_mean': float(wi.mean().item()),
            'w_p50': float(torch.quantile(wi, 0.50).item()),
            'w_p95': float(torch.quantile(wi, 0.95).item()),
            'top1_frac': float(frac[i].item()),
        })

    global_df = pd.DataFrame([{
        'entropy_mean': float(ent.mean().item()),
        'entropy_p50': float(torch.quantile(ent, 0.50).item()),
        'w_max_mean': float(wmax.mean().item()),
        'w_max_p95': float(torch.quantile(wmax, 0.95).item()),
    }])

    return pd.DataFrame(per).sort_values('w_mean', ascending=False), global_df

per_w, global_w = weight_report(w)
print('\n' + '='*100)
print('Gating weight sharpness (test high epoch_1) on (val-like) states')
print('='*100)
display(global_w)
display(per_w)

# ==========================================================
# 5) 组合效果：w 加权 6 个子代理 Q，并补充“极端样本更细归因”
# ==========================================================
@torch.no_grad()
def combine_q(qs_b62, w_b6):
    return (qs_b62 * w_b6.unsqueeze(-1)).sum(dim=1)

@torch.no_grad()
def summarize_q(q_b2, tag=''):
    p = torch.softmax(q_b2, dim=1)
    pmax = p.max(dim=1).values
    gap = (q_b2[:, 0] - q_b2[:, 1]).abs()
    return {
        'tag': tag,
        'q_mean': float(q_b2.mean().item()),
        'q_std': float(q_b2.std().item()),
        'q_min': float(q_b2.min().item()),
        'q_max': float(q_b2.max().item()),
        'gap_mean': float(gap.mean().item()),
        'gap_p95': float(torch.quantile(gap, 0.95).item()),
        'softmax_pmax_mean': float(pmax.mean().item()),
        'softmax_pmax_p95': float(torch.quantile(pmax, 0.95).item()),
    }

w_u = torch.full((B, 6), 1.0/6.0, device=DEVICE)
comb = combine_q(qs, w)
comb_u = combine_q(qs, w_u)

print('\n' + '='*100)
print('Ensemble Q comparison on (val-like) states')
print('='*100)
print(summarize_q(comb, 'weighted(test_high_e1)'))
print(summarize_q(comb_u, 'uniform(1/6)'))

@torch.no_grad()
def top1_hist(w_b6):
    top1 = w_b6.argmax(dim=1)
    counts = torch.bincount(top1, minlength=6).float()
    frac = counts / counts.sum()
    return pd.DataFrame({'agent': names, 'top1_frac': [float(frac[i].item()) for i in range(6)]}).sort_values('top1_frac', ascending=False)

@torch.no_grad()
def extreme_attribution(qs_b62, w_b6, k=12):
    # 以“组合后 gap 最大”的样本为极端样本，输出：
    # - top1 权重是谁
    # - 各 agent 的 gap / qmax
    # - 各 agent 的 w
    comb = combine_q(qs_b62, w_b6)
    comb_gap = (comb[:, 0] - comb[:, 1]).abs()
    idxs = comb_gap.topk(k=min(k, comb_gap.numel())).indices

    gaps = (qs_b62[:, :, 0] - qs_b62[:, :, 1]).abs()            # [B,6]
    qmax = qs_b62.max(dim=2).values                               # [B,6]
    top1 = w_b6.argmax(dim=1)

    rows = []
    for i in idxs.tolist():
        j = int(top1[i].item())
        rows.append({
            'sample_i': i,
            'comb_gap': float(comb_gap[i].item()),
            'top1_agent': names[j],
            'top1_w': float(w_b6[i, j].item()),
            'top1_agent_gap': float(gaps[i, j].item()),
            'top1_agent_qmax': float(qmax[i, j].item()),
            'max_gap_over_agents': float(gaps[i].max().item()),
            'max_qmax_over_agents': float(qmax[i].max().item()),
        })

        # 追加一行“该样本下 6 个 agent 的细节”（便于肉眼定位）
        detail = {
            'sample_i': i,
            'detail': 'w/gap/qmax',
        }
        for a in range(6):
            detail[f'{names[a]}_w'] = float(w_b6[i, a].item())
            detail[f'{names[a]}_gap'] = float(gaps[i, a].item())
            detail[f'{names[a]}_qmax'] = float(qmax[i, a].item())
        rows.append(detail)

    return pd.DataFrame(rows)

print('\n' + '='*100)
print('Attribution on (val-like) states')
print('='*100)
display(top1_hist(w))
display(extreme_attribution(qs, w, k=8))

print('\n' + '='*100)
print('诊断结论怎么看')
print('='*100)
print('1) 如果 top1_frac 仍极度集中（例如只选 vol_1/vol_3），且 entropy 很低 => 高层 gating 塌缩是主要问题。')
print('2) 如果极端样本里 top1_agent 的 gap/qmax 也明显更极端 => 被选中的那个低层 agent 输出在 val 上很不稳。')
print('3) 如果 uniform(1/6) 明显更稳，但 weighted 很差 => gating 的“过度选择”放大了坏代理（更像高层问题）。')


device = cuda:0
N_STATE_1 = 36 N_STATE_2 = 9
[val-sample] groups= 6 total B= 576
tags: ['slope_label1_df1', 'slope_label2_df3', 'slope_label3_df0', 'vol_label1_df1', 'vol_label2_df0', 'vol_label3_df13']
state (576, 36) state_trend (576, 9) previous_action (576,)
loaded test low-level agents

Low-level Q stats on (val-like) states


,agent,q_mean,q_std,q_min,q_max,gap_mean,gap_p95,gap_max
2,slope_3,0.631655,0.087882,0.378615,1.788262,0.046697,0.132283,1.265574
5,vol_3,1.353095,0.133969,0.999251,2.078167,0.228647,0.331539,0.781087
4,vol_2,1.435188,0.133639,1.034090,2.692236,0.203551,0.301790,0.677173
0,slope_1,1.540702,0.177239,0.993373,2.960840,0.267283,0.373929,0.661714
1,slope_2,1.645228,0.148382,1.392179,2.355830,0.263606,0.347583,0.582820
3,vol_1,2.024432,0.230089,1.623694,3.200209,0.263476,0.360958,0.524987


test high gating inferred: N_ACTION= 2 EMB_DIM= 32 HIDDEN= 128 IN_DIM= 64 N_AGENTS= 6
w shape: (576, 6) logits shape: (576, 6)

Gating weight sharpness (test high epoch_1) on (val-like) states


,entropy_mean,entropy_p50,w_max_mean,w_max_p95
0,0.015129,1.381551e-10,0.99394,1.0


,agent,w_mean,w_p50,w_p95,top1_frac
3,vol_1,8.065203e-01,1.000000e+00,1.000000e+00,0.805556
5,vol_3,1.934011e-01,1.283223e-33,1.000000e+00,0.194444
4,vol_2,6.542498e-05,1.340583e-37,1.458735e-05,0.000000
1,slope_2,1.035734e-05,7.258866e-29,4.478837e-07,0.000000
0,slope_1,2.745851e-06,4.477226e-33,2.102974e-08,0.000000
2,slope_3,1.041194e-07,4.484155e-44,1.636363e-11,0.000000



Ensemble Q comparison on (val-like) states
{'tag': 'weighted(test_high_e1)', 'q_mean': 1.9145694971084595, 'q_std': 0.34009939432144165, 'q_min': 1.0933833122253418, 'q_max': 3.200209140777588, 'gap_mean': 0.2586015462875366, 'gap_p95': 0.3609578609466553, 'softmax_pmax_mean': 0.5642222166061401, 'softmax_pmax_p95': 0.5892722606658936}
{'tag': 'uniform(1/6)', 'q_mean': 1.4383835792541504, 'q_std': 0.13505735993385315, 'q_min': 1.2166354656219482, 'q_max': 2.195068836212158, 'gap_mean': 0.20991399884223938, 'gap_p95': 0.27546775341033936, 'softmax_pmax_mean': 0.5522700548171997, 'softmax_pmax_p95': 0.5684347152709961}

Attribution on (val-like) states


,agent,top1_frac
3,vol_1,0.805556
5,vol_3,0.194444
1,slope_2,0.000000
0,slope_1,0.000000
2,slope_3,0.000000
4,vol_2,0.000000


,sample_i,comb_gap,top1_agent,top1_w,top1_agent_gap,top1_agent_qmax,max_gap_over_agents,max_qmax_over_agents,detail,slope_1_w,...,slope_3_qmax,vol_1_w,vol_1_gap,vol_1_qmax,vol_2_w,vol_2_gap,vol_2_qmax,vol_3_w,vol_3_gap,vol_3_qmax
0,510,0.746753,vol_3,0.991905,0.754406,1.839647,0.754406,3.008967,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,510,NaN,NaN,NaN,NaN,NaN,NaN,NaN,w/gap/qmax,6.116544e-05,...,1.362639,5.944144e-04,0.087954,3.008967,7.137447e-03,0.209528,2.014611,9.919046e-01,0.754406,1.839647
2,516,0.524987,vol_1,1.000000,0.524987,3.092691,0.524987,3.092691,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,516,NaN,NaN,NaN,NaN,NaN,NaN,NaN,w/gap/qmax,8.680851e-26,...,0.751867,1.000000e+00,0.524987,3.092691,3.610755e-29,0.290676,1.664505,6.247385e-26,0.250510,1.573561
4,201,0.474898,vol_1,0.889327,0.501186,2.704309,0.582820,2.704309,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,201,NaN,NaN,NaN,NaN,NaN,NaN,NaN,w/gap/qmax,8.919059e-04,...,1.330730,8.893265e-01,0.501186,2.704309,2.312583e-03,0.086618,2.031006,1.046034e-01,0.258905,1.914530
6,552,0.468618,vol_3,0.999998,0.468619,1.588368,0.468619,2.360194,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,552,NaN,NaN,NaN,NaN,NaN,NaN,NaN,w/gap/qmax,1.026683e-11,...,0.734491,1.025782e-08,0.275916,2.360194,1.770094e-06,0.309309,1.636194,9.999982e-01,0.468619,1.588368
8,560,0.439117,vol_1,1.000000,0.439117,2.343569,0.439117,2.343569,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,560,NaN,NaN,NaN,NaN,NaN,NaN,NaN,w/gap/qmax,5.915380e-18,...,1.052402,1.000000e+00,0.439117,2.343569,1.874137e-20,0.214619,1.724871,4.174074e-18,0.265088,1.698705



诊断结论怎么看
1) 如果 top1_frac 仍极度集中（例如只选 vol_1/vol_3），且 entropy 很低 => 高层 gating 塌缩是主要问题。
2) 如果极端样本里 top1_agent 的 gap/qmax 也明显更极端 => 被选中的那个低层 agent 输出在 val 上很不稳。
3) 如果 uniform(1/6) 明显更稳，但 weighted 很差 => gating 的“过度选择”放大了坏代理（更像高层问题）。


In [3]:
# ==========================================================
# 6) 离线实验：不重训，直接在 val-like states 上对 gating 权重做“温度/地板”处理
#    目的：快速验证“soften gating”能否显著改善权重塌缩与 ensemble Q 极端程度
# ==========================================================
import itertools

@torch.no_grad()
def soften_w(w_b6: torch.Tensor, T: float = 1.0, floor: float = 0.0):
    """对概率 w 做温度 softmax(log(w)/T) + floor(prob) 后重归一化。"""
    ww = w_b6.clamp_min(1e-8)
    if T is not None and float(T) != 1.0:
        ww = torch.softmax(torch.log(ww) / float(T), dim=1)
    if floor is not None and float(floor) > 0.0:
        ww = ww + float(floor)
        ww = ww / ww.sum(dim=1, keepdim=True)
    return ww

@torch.no_grad()
def gating_summary(w_b6: torch.Tensor):
    eps = 1e-12
    ent = -(w_b6.clamp_min(eps) * w_b6.clamp_min(eps).log()).sum(dim=1)
    wmax = w_b6.max(dim=1).values
    top1 = w_b6.argmax(dim=1)
    counts = torch.bincount(top1, minlength=6).float()
    frac = counts / counts.sum()
    return {
        'entropy_mean': float(ent.mean().item()),
        'entropy_p50': float(torch.quantile(ent, 0.50).item()),
        'w_max_mean': float(wmax.mean().item()),
        'w_max_p95': float(torch.quantile(wmax, 0.95).item()),
        **{f'top1_{names[i]}': float(frac[i].item()) for i in range(6)},
    }

@torch.no_grad()
def ensemble_summary(qs_b62: torch.Tensor, w_b6: torch.Tensor, tag: str):
    qq = combine_q(qs_b62, w_b6)
    s = summarize_q(qq, tag)
    # 额外给一个更直观的“极端比例”：gap > p95(uniform) 多少
    return s

# 扫描网格（可按需要再调细）
T_list = [1.0, 1.5, 2.0, 3.0]
floor_list = [0.0, 0.001, 0.005, 0.01]

rows = []
for T, floor in itertools.product(T_list, floor_list):
    w_tf = soften_w(w, T=T, floor=floor)
    g = gating_summary(w_tf)
    e = ensemble_summary(qs, w_tf, tag=f'T={T},floor={floor}')
    rows.append({
        'T': T,
        'floor': floor,
        **{k: g[k] for k in ['entropy_mean','w_max_mean']},
        'gap_mean': e['gap_mean'],
        'gap_p95': e['gap_p95'],
        'softmax_pmax_mean': e['softmax_pmax_mean'],
        'softmax_pmax_p95': e['softmax_pmax_p95'],
        # 看 top1 是否还“塌缩到少数 agent”
        'top1_slope_sum': g['top1_slope_1'] + g['top1_slope_2'] + g['top1_slope_3'],
        'top1_vol_sum': g['top1_vol_1'] + g['top1_vol_2'] + g['top1_vol_3'],
        'top1_vol_1': g['top1_vol_1'],
        'top1_vol_3': g['top1_vol_3'],
    })

report = pd.DataFrame(rows)
report = report.sort_values(['gap_p95','softmax_pmax_p95'], ascending=[True, True])

print('\n' + '='*100)
print('Offline gating soften grid search (smaller gap_p95 & pmax_p95 is usually better / more stable)')
print('='*100)
display(report)

# 挑出最稳的几个候选，再输出它们的 top1 直方图，方便你肉眼对比
best = report.head(5)[['T','floor']].to_dict('records')
print('Top-5 candidates:', best)
for cfg in best:
    w_tf = soften_w(w, T=cfg['T'], floor=cfg['floor'])
    print('\n---', cfg, '---')
    display(top1_hist(w_tf))

# 也给一个“和 uniform 的相对差距”参考
uniform_stats = summarize_q(comb_u, 'uniform(1/6)')
print('\n[ref] uniform stats:', uniform_stats)



Offline gating soften grid search (smaller gap_p95 & pmax_p95 is usually better / more stable)


,T,floor,entropy_mean,w_max_mean,gap_mean,gap_p95,softmax_pmax_mean,softmax_pmax_p95,top1_slope_sum,top1_vol_sum,top1_vol_1,top1_vol_3
15,3.0,0.010,0.360816,0.922010,0.254742,0.351503,0.563288,0.586982,0.0,1.0,0.805556,0.194444
11,2.0,0.010,0.298381,0.939420,0.255562,0.354168,0.563485,0.587628,0.0,1.0,0.805556,0.194444
7,1.5,0.010,0.286908,0.943666,0.255715,0.354239,0.563520,0.587645,0.0,1.0,0.805556,0.194444
3,1.0,0.010,0.278738,0.947113,0.255791,0.354242,0.563537,0.587646,0.0,1.0,0.805556,0.194444
14,3.0,0.005,0.259450,0.944011,0.256076,0.355133,0.563614,0.587862,0.0,1.0,0.805556,0.194444
10,2.0,0.005,0.188466,0.961927,0.256919,0.356877,0.563817,0.588284,0.0,1.0,0.805556,0.194444
6,1.5,0.005,0.175799,0.966297,0.257077,0.356954,0.563853,0.588303,0.0,1.0,0.805556,0.194444
2,1.0,0.005,0.166960,0.969845,0.257155,0.356957,0.563870,0.588304,0.0,1.0,0.805556,0.194444
13,3.0,0.001,0.162483,0.962556,0.257200,0.357561,0.563889,0.588450,0.0,1.0,0.805556,0.194444
12,3.0,0.000,0.134561,0.967331,0.257489,0.358315,0.563959,0.588633,0.0,1.0,0.805556,0.194444


Top-5 candidates: [{'T': 3.0, 'floor': 0.01}, {'T': 2.0, 'floor': 0.01}, {'T': 1.5, 'floor': 0.01}, {'T': 1.0, 'floor': 0.01}, {'T': 3.0, 'floor': 0.005}]

--- {'T': 3.0, 'floor': 0.01} ---


,agent,top1_frac
3,vol_1,0.805556
5,vol_3,0.194444
1,slope_2,0.000000
0,slope_1,0.000000
2,slope_3,0.000000
4,vol_2,0.000000



--- {'T': 2.0, 'floor': 0.01} ---


,agent,top1_frac
3,vol_1,0.805556
5,vol_3,0.194444
1,slope_2,0.000000
0,slope_1,0.000000
2,slope_3,0.000000
4,vol_2,0.000000


... (3 outputs left out)

,agent,top1_frac
3,vol_1,0.805556
5,vol_3,0.194444
1,slope_2,0.000000
0,slope_1,0.000000
2,slope_3,0.000000
4,vol_2,0.000000



--- {'T': 3.0, 'floor': 0.005} ---


,agent,top1_frac
3,vol_1,0.805556
5,vol_3,0.194444
1,slope_2,0.000000
0,slope_1,0.000000
2,slope_3,0.000000
4,vol_2,0.000000



[ref] uniform stats: {'tag': 'uniform(1/6)', 'q_mean': 1.4383835792541504, 'q_std': 0.13505735993385315, 'q_min': 1.2166354656219482, 'q_max': 2.195068836212158, 'gap_mean': 0.20991399884223938, 'gap_p95': 0.27546775341033936, 'softmax_pmax_mean': 0.5522700548171997, 'softmax_pmax_p95': 0.5684347152709961}


In [5]:
# ==========================================================
# 7) 诊断：坍缩是否由“低层 Q 天然占优”驱动？
#    (A) argmax_a qmax(a) 的分布
#    (B) gating top1 vs qmax-argmax 的一致率
#    (C) slope vs vol 的 qmax 分布对比（均值/分位数）
# ==========================================================

@torch.no_grad()
def qmax_and_gap(qs_b62: torch.Tensor):
    qmax = qs_b62.max(dim=2).values          # [B,6]
    gap = (qs_b62[:, :, 0] - qs_b62[:, :, 1]).abs()  # [B,6]
    return qmax, gap

@torch.no_grad()
def hist_frac_from_argmax(argmax_idx: torch.Tensor, n: int):
    counts = torch.bincount(argmax_idx, minlength=n).float()
    frac = counts / counts.sum().clamp_min(1.0)
    return pd.DataFrame({'agent': names, 'frac': [float(frac[i].item()) for i in range(n)]}).sort_values('frac', ascending=False)

qmax_b6, gap_b6 = qmax_and_gap(qs)

# (A) 哪个 agent 在 qmax 上“天然最强”
qmax_top1 = qmax_b6.argmax(dim=1)  # [B]
print('\n' + '='*100)
print('A) Distribution of argmax_a qmax(a) (which low-level agent is strongest per-sample)')
print('='*100)
display(hist_frac_from_argmax(qmax_top1, 6))

# (B) gating top1 与 qmax_top1 的一致率
w_top1 = w.argmax(dim=1)
agree = (w_top1 == qmax_top1).float().mean().item()
print('\n' + '='*100)
print('B) Agreement between gating top1 and qmax-argmax')
print('='*100)
print('agreement_rate =', agree)

# (C) slope vs vol 的 qmax 分布对比
slope_cols = [0, 1, 2]
vol_cols = [3, 4, 5]

qmax_slope = qmax_b6[:, slope_cols].max(dim=1).values  # 每个样本：slope 组里最强的 qmax
qmax_vol = qmax_b6[:, vol_cols].max(dim=1).values      # 每个样本：vol 组里最强的 qmax

rows = []
for name_, vec in [('best_slope_qmax', qmax_slope), ('best_vol_qmax', qmax_vol)]:
    rows.append({
        'group': name_,
        'mean': float(vec.mean().item()),
        'p10': float(torch.quantile(vec, 0.10).item()),
        'p50': float(torch.quantile(vec, 0.50).item()),
        'p90': float(torch.quantile(vec, 0.90).item()),
        'p95': float(torch.quantile(vec, 0.95).item()),
    })

print('\n' + '='*100)
print('C) qmax comparison: best slope-agent vs best vol-agent per-sample')
print('='*100)
display(pd.DataFrame(rows))

# 还可以看：vol 是否不仅 qmax 大，而且 gap 更大（更“自信/尖锐”）
gap_slope = gap_b6[:, slope_cols].max(dim=1).values
gap_vol = gap_b6[:, vol_cols].max(dim=1).values
rows2 = []
for name_, vec in [('best_slope_gap', gap_slope), ('best_vol_gap', gap_vol)]:
    rows2.append({
        'group': name_,
        'mean': float(vec.mean().item()),
        'p50': float(torch.quantile(vec, 0.50).item()),
        'p90': float(torch.quantile(vec, 0.90).item()),
        'p95': float(torch.quantile(vec, 0.95).item()),
        'max': float(vec.max().item()),
    })

print('\n' + '='*100)
print('D) gap comparison: best slope-agent vs best vol-agent per-sample')
print('='*100)
display(pd.DataFrame(rows2))

print('\n解释怎么用：')
print('1) 如果 A) 里 qmax_top1 也高度集中在 vol_1/vol_3，同时 C) 显示 best_vol_qmax 显著高于 best_slope_qmax，说明“低层天然占优”确实在驱动坍缩。')
print('2) 如果 B) agreement_rate 很高（接近 1），说明 gating 只是学会了挑 qmax 最大的那个：训练上看似合理，但对分布漂移会非常脆弱。')
print('3) 如果 D) best_vol_gap 也显著更大，说明 vol 组不仅值大而且更尖锐，更容易把 TD/KL 目标推向 one-hot。')



A) Distribution of argmax_a qmax(a) (which low-level agent is strongest per-sample)


,agent,frac
3,vol_1,0.996528
0,slope_1,0.003472
1,slope_2,0.000000
2,slope_3,0.000000
4,vol_2,0.000000
5,vol_3,0.000000



B) Agreement between gating top1 and qmax-argmax
agreement_rate = 0.8020833134651184

C) qmax comparison: best slope-agent vs best vol-agent per-sample


,group,mean,p10,p50,p90,p95
0,best_slope_qmax,1.781953,1.720236,1.764358,1.837457,1.872173
1,best_vol_qmax,2.156170,1.955524,2.115288,2.383081,2.536644



D) gap comparison: best slope-agent vs best vol-agent per-sample


,group,mean,p50,p90,p95,max
0,best_slope_gap,0.293837,0.282791,0.350045,0.384764,1.265574
1,best_vol_gap,0.282538,0.274014,0.356170,0.382319,0.781087



解释怎么用：
1) 如果 A) 里 qmax_top1 也高度集中在 vol_1/vol_3，同时 C) 显示 best_vol_qmax 显著高于 best_slope_qmax，说明“低层天然占优”确实在驱动坍缩。
2) 如果 B) agreement_rate 很高（接近 1），说明 gating 只是学会了挑 qmax 最大的那个：训练上看似合理，但对分布漂移会非常脆弱。
3) 如果 D) best_vol_gap 也显著更大，说明 vol 组不仅值大而且更尖锐，更容易把 TD/KL 目标推向 one-hot。
